In [1]:
# First, install everything (run this cell first)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate safetensors
!pip install -q openai pillow matplotlib numpy pandas tqdm
!pip install -q opencv-python imageio imageio-ffmpeg
!pip install -q bayesian-optimization scikit-learn
!pip install -q flask flask-cors pyngrok

print("✅ All packages installed!")

✅ All packages installed!


In [ ]:
# =============================================================================
# CONFIGURATION — EDIT YOUR API KEYS HERE ONLY
# =============================================================================

CONFIG = {
    "DEEPSEEK_API_KEY": "xxxxx",
    "NGROK_AUTH_TOKEN": "xxxxx",
}

print("✅ Configuration loaded")

✅ Configuration loaded


In [ ]:
# =============================================================================
# TEXT-TO-VIDEO SERVER - GOOGLE COLAB VERSION
# Converts existing code into a web server
# =============================================================================



import gc
import torch
import json
import re
import time
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from diffusers import DiffusionPipeline, EulerDiscreteScheduler
from diffusers import StableDiffusionXLImg2ImgPipeline, StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from openai import OpenAI
import os
import subprocess
import shutil
import tempfile
import urllib.request
import imageio
import glob
from datetime import datetime
import threading

# Flask imports for server
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok

# Import Bayesian optimization with fallback
try:
    from bayes_opt import BayesianOptimization
    BAYESIAN_AVAILABLE = True
    print("✅ Bayesian optimization available")
except ImportError:
    print("⚠️ Bayesian optimization not available, using fallback method")
    BAYESIAN_AVAILABLE = False

# =============================================================================
# EXISTING UTILITY FUNCTIONS
# =============================================================================

def clear_gpu_memory():
    """Clear GPU memory and cache."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def setup_ffmpeg():
    """Install and setup ffmpeg for video processing"""
    try:
        subprocess.run(['ffmpeg', '-version'], capture_output=True, check=True)
        print("FFmpeg is already installed")
    except:
        print("Installing FFmpeg...")
        os.system("apt update -qq")
        os.system("apt install -y -qq ffmpeg")
        print("FFmpeg installed successfully")

def create_optimized_prompt_for_sd(prompt, max_tokens=75):
    """Use AI to optimize and condense prompts for Stable Diffusion within token limits"""
    if len(prompt.split()) <= max_tokens:
        return prompt

    try:
        from openai import OpenAI
        client = OpenAI(
            api_key=CONFIG["DEEPSEEK_API_KEY"],
            base_url="https://api.deepseek.com"
        )

        optimization_prompt = f"""Optimize this Stable Diffusion prompt to be under {max_tokens} words while preserving all key visual information:

Original prompt: "{prompt}"

Requirements:
1. Keep under {max_tokens} words exactly
2. Preserve all character descriptions (age, clothing, appearance)
3. Preserve all setting/location details
4. Preserve all actions and poses
5. Remove redundant words but keep visual clarity
6. Use comma-separated format optimal for Stable Diffusion
7. Prioritize: character → action → setting → style

Output ONLY the optimized prompt, nothing else."""

        response = client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": optimization_prompt}]
        )

        optimized = response.choices[0].message.content.strip()

        words = optimized.split()
        if len(words) > max_tokens:
            optimized = " ".join(words[:max_tokens])

        return optimized

    except Exception as e:
        print(f"Prompt optimization failed, using truncation: {e}")
        words = prompt.split()
        return " ".join(words[:max_tokens])



class EnhancedTextAnalyzer:
    def __init__(self, api_key=None):
        if api_key is None:
            api_key = CONFIG["DEEPSEEK_API_KEY"]
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        self.chunk_cache = {}

    def segment_text_intelligently(self, text, max_scenes=6):
        """Convert any text into logical visual scenes for video generation"""
        prompt = f"""Please analyze this text and convert it into {max_scenes} distinct visual scenes for video generation.

        Input text: "{text}"

        Requirements:
        1. Create exactly {max_scenes} scenes that tell a coherent story
        2. Each scene should feature THE SAME MAIN CHARACTER/SUBJECT throughout
        3. Maintain visual consistency - same person, same general setting/world
        4. Create a logical progression that flows smoothly between scenes
        5. Each scene should be 1-2 sentences describing a specific visual moment
        6. Focus on concrete, filmable actions and settings
        7. Ensure continuity of character appearance, clothing, and environment

        Output format:
        Scene 1: [concrete visual description with consistent character]
        Scene 2: [same character, logical progression]
        Scene 3: [continuing the story with same character]
        ...

        Example for "a man jumping on grassland":
        Scene 1: A young athletic man in blue athletic wear stands ready on a vast green grassland, muscles tensed for action
        Scene 2: The same man launches powerfully into the air above the grassland, arms spread wide, hair flowing in the wind
        Scene 3: The man lands gracefully back on the green grass, slightly crouched, with a satisfied expression on his face
        """

        response = self.client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": prompt}]
        )

        scenes = []
        for line in response.choices[0].message.content.strip().split('\n'):
            if line.startswith('Scene '):
                scene_text = line.split(':', 1)[1].strip() if ':' in line else line
                scenes.append(scene_text)

        return scenes[:max_scenes]

    def translate_to_english(self, text):
        """Original translation function - kept for consistency"""
        if self._is_primarily_english(text):
            return text

        prompt = f"""Please translate the following text to English, maintaining the visual and descriptive nature of the content:

        Text: {text}

        Requirements:
        1. Translate to natural, fluent English
        2. Preserve all visual descriptions and imagery
        3. Keep any technical or specific terms
        4. Maintain the original structure where applicable
        """

        response = self.client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    def _is_primarily_english(self, text):
        """Check if text is primarily in English"""
        english_chars = sum(1 for c in text if c.isascii() and c.isalpha())
        total_chars = sum(1 for c in text if c.isalpha())
        return total_chars == 0 or english_chars / total_chars > 0.7

    def interpret_cultural_terms(self, text, context):
        """Original cultural interpretation function"""
        prompt = f"""Please analyze this text for any specialized terms, cultural references, or abstract concepts, and convert them to concrete visual descriptions:

        Context: {context}
        Current text: {text}

        Please identify any abstract or specialized terms and provide concrete visual descriptions for them.
        Format: term: visual description

        If no specialized terms exist, just return the original text with enhanced visual details.
        """

        response = self.client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": prompt}]
        )

        interpretations = {}
        result_text = response.choices[0].message.content.strip()

        for line in result_text.split('\n'):
            if ':' in line and not line.startswith('Context') and not line.startswith('Current'):
                try:
                    term, desc = line.split(':', 1)
                    interpretations[term.strip()] = desc.strip()
                except:
                    continue

        interpreted_text = text
        for term, desc in interpretations.items():
            interpreted_text = interpreted_text.replace(term, desc)

        return interpreted_text if interpretations else text

    def get_text_understanding(self, full_text):
        """Original understanding function adapted for general text"""
        study_prompt = f"""Please analyze this text from a visual perspective, describing:
        1. Main themes and concepts that can be visualized
        2. Emotional tone and atmosphere
        3. Key visual elements and imagery
        4. Suggested visual style and mood
        5. Color palette and lighting suggestions

        Text: {full_text}

        Please respond in English with concrete visual language, avoiding abstract concepts.
        """

        response = self.client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": study_prompt}]
        )

        return response.choices[0].message.content

    def analyze_chunk_detail(self, chunk, category, context_dict):
        """Original detailed analysis function """
        cache_key = f"{chunk}_{category}"
        if cache_key in self.chunk_cache:
            return self.chunk_cache[cache_key]

        english_translation = self.translate_to_english(chunk)
        interpreted_chunk = self.interpret_cultural_terms(chunk, context_dict.get('full_text', ''))
        text_understanding = self.get_text_understanding(context_dict.get('full_text', ''))
        previous_chunk = context_dict.get('previous_chunk', '')

        prompts = {
            "subject_action": f"""
            Previous analysis:
            {english_translation}
            {interpreted_chunk}
            {text_understanding}

            Analyze the subjects and their actions in this scene: "{chunk}"
            Previous scene: "{previous_chunk}"

            Return in this exact format:
            subjects: [concrete description of each person/animal/living being, their clothing/appearance]
            actions: [specific descriptions of actions or emotional states]

            Requirements:
            - All descriptions must be visually concrete
            - Include full description of subjects
            - If no explicit subject in current scene, use subject from previous scene
            - All actions must be visually concrete
            - Avoid abstract descriptions
            - Do not mention if something is "not described"
            - List each subject and action exactly once
            - Use concise, visual descriptions
            """,

            "scene_setting": f"""
            Previous analysis:
            {english_translation}
            {interpreted_chunk}
            {text_understanding}

            Analyze the scene and environmental elements in: "{chunk}"

            Return in this format:
            locations: [specific scene locations]
            objects: [specific physical objects, flora, fauna, architectural elements]

            Requirements:
            - Descriptions must be concrete and visual
            - Include all physical elements mentioned or implied
            - Be specific about quantities and types
            - List each element exactly once
            - Use concise, visual terms
            """,

            "time_weather": f"""
            Previous analysis:
            {english_translation}
            {interpreted_chunk}
            {text_understanding}

            Analyze time and weather elements in: "{chunk}"

            Return in this format:
            time: [specific time, e.g., sunset, dawn, night]
            weather: [specific weather conditions]

            Requirements:
            - Use commonly recognized natural phenomena
            - Be specific about timing
            - Include atmospheric conditions
            - List each element exactly once
            """,

            "mood": f"""
            Previous analysis:
            {english_translation}
            {interpreted_chunk}
            {text_understanding}

            Analyze visual atmosphere and mood in: "{chunk}"

            Return in this format:
            lighting: [specific lighting effects]
            atmosphere: [specific visual mood]
            color_tone: [main color tones]

            Requirements:
            - All descriptions should be directly usable for image generation
            - Transform abstract emotions into visual metaphors
            - Include seasonal or temporal color palettes
            - Use concise, visual terms
            """
        }

        response = self.client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": prompts[category]}]
        )

        result = response.choices[0].message.content.strip()
        self.chunk_cache[cache_key] = result
        return result

    def analyze_chunk_parallel(self, chunk, context_dict):
        """Original parallel analysis function - preserved"""
        try:
            print(f"\nAnalyzing scene: {chunk}")

            with ThreadPoolExecutor(max_workers=4) as executor:
                futures = {
                    "subject_action": executor.submit(self.analyze_chunk_detail, chunk, "subject_action", context_dict),
                    "scene_setting": executor.submit(self.analyze_chunk_detail, chunk, "scene_setting", context_dict),
                    "time_weather": executor.submit(self.analyze_chunk_detail, chunk, "time_weather", context_dict),
                    "mood": executor.submit(self.analyze_chunk_detail, chunk, "mood", context_dict)
                }

                results = {"text": chunk}

                for category, future in futures.items():
                    try:
                        result = future.result(timeout=60)
                        if result:
                            results[category] = result
                    except Exception as e:
                        print(f"Error getting result for {category}: {e}")
                        results[category] = ""

                return results

        except Exception as e:
            print(f"Error in analyze_chunk_parallel: {str(e)}")
            return {"text": chunk}

    def pack_chunk_to_prompt(self, chunk_analysis, overall_understanding, context_dict=None):
        """Original prompt packing function - preserved"""
        elements = {
            'primary_subjects': [],
            'secondary_subjects': [],
            'actions': [],
            'objects': [],
            'environment': [],
            'lighting': [],
            'atmosphere': [],
            'color_tone': [],
            'weather': [],
            'time': [],
            'style': ['cinematic photography', 'high quality', 'detailed']
        }

        # Parse subject_action
        if 'subject_action' in chunk_analysis:
            content = chunk_analysis['subject_action']
            if 'subjects:' in content and 'actions:' in content:
                lines = content.split('\n')
                for line in lines:
                    if line.startswith('subjects:'):
                        subjects = line.replace('subjects:', '').strip()
                        elements['primary_subjects'].append(subjects)
                    elif line.startswith('actions:'):
                        actions = line.replace('actions:', '').strip()
                        elements['actions'].append(actions)

        # Parse scene_setting
        if 'scene_setting' in chunk_analysis:
            content = chunk_analysis['scene_setting']
            for line in content.split('\n'):
                if line.startswith('locations:'):
                    locations = line.replace('locations:', '').strip()
                    if locations:
                        elements['environment'].append(locations)
                elif line.startswith('objects:'):
                    objects = line.replace('objects:', '').strip()
                    if objects:
                        elements['objects'].append(objects)

        # Parse time_weather
        if 'time_weather' in chunk_analysis:
            content = chunk_analysis['time_weather']
            for line in content.split('\n'):
                if line.startswith('time:'):
                    time = line.replace('time:', '').strip()
                    if time:
                        elements['time'].append(time)
                elif line.startswith('weather:'):
                    weather = line.replace('weather:', '').strip()
                    if weather:
                        elements['weather'].append(weather)

        # Parse mood
        if 'mood' in chunk_analysis:
            content = chunk_analysis['mood']
            for line in content.split('\n'):
                if line.startswith('lighting:'):
                    lighting = line.replace('lighting:', '').strip()
                    if lighting:
                        elements['lighting'].append(lighting)
                elif line.startswith('atmosphere:'):
                    atmosphere = line.replace('atmosphere:', '').strip()
                    if atmosphere:
                        elements['atmosphere'].append(atmosphere)
                elif line.startswith('color_tone:'):
                    color_tone = line.replace('color_tone:', '').strip()
                    if color_tone:
                        elements['color_tone'].append(color_tone)

        # Build the prompt
        prompt_parts = []

        if elements['primary_subjects']:
            subjects = elements['primary_subjects'][0].split(',')[0]
            prompt_parts.append(f"featuring {subjects}")

        if elements['actions']:
            actions = elements['actions'][0].split(',')[0]
            prompt_parts.append(f"engaged in {actions}")

        if elements['environment']:
            env = elements['environment'][0].split(',')[0]
            prompt_parts.append(f"set in {env}")

        if elements['objects']:
            objects = elements['objects'][0].split(',')[0]
            prompt_parts.append(f"with {objects}")

        if elements['weather'] or elements['lighting']:
            conditions = []
            if elements['weather']:
                conditions.append(elements['weather'][0].split(',')[0])
            if elements['lighting']:
                conditions.append(elements['lighting'][0].split(',')[0])
            if conditions:
                prompt_parts.append(f"under {', '.join(conditions)}")

        if elements['atmosphere']:
            atmosphere = elements['atmosphere'][0].split(',')[0]
            prompt_parts.append(f"creating {atmosphere} atmosphere")

        if elements['color_tone']:
            color = elements['color_tone'][0].split(',')[0]
            prompt_parts.append(f"in {color}")

        prompt_parts.extend(elements['style'])

        return ', '.join(filter(None, prompt_parts))

    def get_contextual_analysis(self, full_text, segment, context_dict):
        """Original contextual analysis function - preserved"""
        try:
            context_dict['current_chunk'] = segment
            overall_understanding = self.get_text_understanding(full_text)
            detailed_analysis = self.analyze_chunk_parallel(segment, context_dict)

            compact_prompt = self.pack_chunk_to_prompt(
                detailed_analysis,
                overall_understanding,
                context_dict
            )

            return {
                "original_text": full_text,
                "translated_text": self.translate_to_english(full_text),
                "overall_understanding": overall_understanding,
                "detailed_analysis": detailed_analysis,
                "compact_prompt": compact_prompt
            }
        except Exception as e:
            print(f"Error in get_contextual_analysis: {str(e)}")
            return None

    def create_consistent_prompt(self, scene_text, scene_index, total_scenes, base_character_description=""):
      """Create a consistent prompt that maintains character and setting continuity for Stable Diffusion"""

      if scene_index == 0:
          # First scene - establish the character with SD-specific guidance
          consistency_prompt = f"""Create a Stable Diffusion prompt for this scene: "{scene_text}"

  IMPORTANT: Stable Diffusion cannot reference "same person" or "previous image" - you must describe the character fully each time.

  Create a detailed character that will be consistent across multiple scenes:

  Requirements:
  1. Under 65 words total
  2. Format: [detailed character: age, build, hair, clothing], [specific action], [setting], [lighting/style]
  3. Be very specific about character appearance for consistency
  4. Use concrete visual terms Stable Diffusion understands
  5. Include "cinematic photography, high quality" at end

  Example format: "A tall athletic man in his 20s with short brown hair wearing grey t-shirt and dark jeans, jumping with arms raised, on sunny green grassland, golden hour lighting, cinematic photography, high quality"

  Output ONLY the prompt, nothing else."""

          response = self.client.chat.completions.create(
              model="deepseek-v4-flash",
              messages=[{"role": "user", "content": consistency_prompt}]
          )

          optimized_prompt = response.choices[0].message.content.strip()

          # Extract character description for future use - be more flexible
          character_parts = optimized_prompt.split(',')
          if len(character_parts) >= 1:
              # Take the character description (usually the first part before first comma)
              base_character_description = character_parts[0].strip()
              # Clean up if it starts with "A " or "An "
              if base_character_description.lower().startswith(('a ', 'an ')):
                  base_character_description = base_character_description[2:].strip()
          else:
              base_character_description = "athletic man in grey t-shirt"

          return optimized_prompt, base_character_description

      else:
          # Subsequent scenes - maintain consistency with SD-specific approach
          consistency_prompt = f"""Create a Stable Diffusion prompt for this scene: "{scene_text}"

  CRITICAL: Stable Diffusion cannot understand "same person" - you must fully redescribe the character.

  Character to maintain: "{base_character_description}"
  New scene: "{scene_text}"

  Requirements:
  1. Under 65 words total
  2. Start with EXACT character description: "{base_character_description}"
  3. Add the new action from the scene
  4. Keep same setting/environment style for continuity
  5. Format: [exact character], [new action], [consistent setting], [cinematic photography, high quality]
  6. NO references to "same person" or "previous scene"

  Example: "A tall athletic man in his 20s with short brown hair wearing grey t-shirt and dark jeans, landing gracefully after jump, on sunny green grassland, golden hour lighting, cinematic photography, high quality"

  Output ONLY the prompt, nothing else."""

          response = self.client.chat.completions.create(
              model="deepseek-v4-flash",
              messages=[{"role": "user", "content": consistency_prompt}]
          )

          result = response.choices[0].message.content.strip()
          return result, base_character_description

    def analyze_text_to_consistent_prompts(self, text, max_scenes=6):
        """Main pipeline: text -> consistent scenes -> optimized prompts"""
        print(f"Segmenting text into {max_scenes} consistent scenes...")
        scenes = self.segment_text_intelligently(text, max_scenes)

        print(f"Generated {len(scenes)} scenes:")
        for i, scene in enumerate(scenes):
            print(f"  Scene {i+1}: {scene}")

        scene_data = []
        base_character_description = ""

        for i, scene in enumerate(scenes):
            print(f"\nOptimizing scene {i+1} for consistency...")

            if i == 0:
                optimized_prompt, base_character_description = self.create_consistent_prompt(
                    scene, i, len(scenes)
                )
            else:
                optimized_prompt, _ = self.create_consistent_prompt(
                    scene, i, len(scenes), base_character_description
                )

            # Truncate for CLIP compatibility
            final_prompt = create_optimized_prompt_for_sd(optimized_prompt)

            scene_data.append({
                'scene_text': scene,
                'image_prompt': final_prompt,
                'scene_index': i,
                'character_description': base_character_description
            })

            print(f"Optimized prompt: {final_prompt}")

        return scene_data

    def analyze_text_to_prompts(self, text, max_scenes=6):
        """Legacy method for backward compatibility - uses original ontology"""
        print(f"Segmenting text into {max_scenes} scenes...")
        scenes = self.segment_text_intelligently(text, max_scenes)

        print(f"Generated {len(scenes)} scenes:")
        for i, scene in enumerate(scenes):
            print(f"  Scene {i+1}: {scene}")

        scene_data = []
        for i, scene in enumerate(scenes):
            print(f"\nAnalyzing scene {i+1} with full ontology...")

            context_dict = {
                'full_text': text,
                'previous_chunk': scenes[i-1] if i > 0 else None,
                'chunk_index': i,
                'total_scenes': len(scenes)
            }

            analysis = self.get_contextual_analysis(text, scene, context_dict)

            if analysis:
                # Truncate for CLIP compatibility
                final_prompt = create_optimized_prompt_for_sd(analysis['compact_prompt'])

                scene_data.append({
                    'scene_text': scene,
                    'analysis': analysis,
                    'image_prompt': final_prompt,
                    'scene_index': i
                })
                print(f"Generated prompt: {final_prompt[:100]}...")
            else:
                print(f"Failed to analyze scene {i+1}")

        return scene_data

# BayesianStableDiffusion class exactly
class BayesianStableDiffusion:
    def __init__(self, model_id="stabilityai/stable-diffusion-xl-base-1.0", num_inference_steps=50,
                 clip_model_name="openai/clip-vit-base-patch32"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_id = model_id
        self.refiner_id = "stabilityai/stable-diffusion-xl-refiner-1.0"

        print(f"Initializing models on device: {self.device}")
        if torch.cuda.is_available():
            print(f"Available CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
            clear_gpu_memory()

        try:
            print(f"Loading base model {model_id}...")
            self.base = DiffusionPipeline.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                variant="fp16",
                use_safetensors=True
            ).to(self.device)

            print(f"Loading refiner model...")
            self.refiner = DiffusionPipeline.from_pretrained(
                self.refiner_id,
                torch_dtype=torch.float16,
                variant="fp16",
                use_safetensors=True,
                text_encoder_2=self.base.text_encoder_2,
                vae=self.base.vae,
            ).to(self.device)

            self.base.scheduler = EulerDiscreteScheduler.from_config(
                self.base.scheduler.config,
                use_karras_sigmas=True
            )
            self.refiner.scheduler = EulerDiscreteScheduler.from_config(
                self.refiner.scheduler.config,
                use_karras_sigmas=True
            )

            for pipe in [self.base, self.refiner]:
                try:
                    pipe.enable_attention_slicing(slice_size="auto")
                    pipe.enable_vae_slicing()
                    pipe.enable_xformers_memory_efficient_attention()
                except Exception as e:
                    print(f"Warning: Could not enable some optimizations: {e}")

            print("Loading CLIP model...")
            self.num_inference_steps = num_inference_steps
            self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
            self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(self.device)
            self.clip_model.eval()

            print("Model initialization completed")

        except Exception as e:
            print(f"Error initializing model: {str(e)}")
            traceback.print_exc()
            raise

    def generate_images(self, prompt, negative_prompt="", num_samples=5, guidance_scale=9, temperature=1.0, seed=None):
      #Generate images with optional seed for consistency
      try:
          clear_gpu_memory()

          # Set seed for reproducibility
          if seed is not None:
              torch.manual_seed(seed)
              if torch.cuda.is_available():
                  torch.cuda.manual_seed(seed)
                  torch.cuda.manual_seed_all(seed)
              np.random.seed(seed)
              # Set generator for the pipeline
              generator = torch.Generator(device=self.device).manual_seed(seed)
          else:
              generator = None

          prompt_truncated = create_optimized_prompt_for_sd(prompt)
          negative_prompt_truncated = create_optimized_prompt_for_sd(negative_prompt)

          print(f"Generating {num_samples} images with seed: {seed}")

          base_images = self.base(
              prompt=[prompt_truncated] * num_samples,
              negative_prompt=[negative_prompt_truncated] * num_samples,
              num_inference_steps=30,
              denoising_end=0.8,
              guidance_scale=guidance_scale,
              width=1024,
              height=1024,
              generator=generator,  # Add generator here
          ).images

          refined_images = []
          for i, base_image in enumerate(base_images):
              # Use incremental seed for variety within same character
              if seed is not None:
                  refiner_generator = torch.Generator(device=self.device).manual_seed(seed + i)
              else:
                  refiner_generator = None

              refined = self.refiner(
                  prompt=prompt_truncated,
                  negative_prompt=negative_prompt_truncated,
                  image=base_image,
                  num_inference_steps=20,
                  denoising_start=0.8,
                  guidance_scale=guidance_scale,
                  generator=refiner_generator,  # Add generator here too
              ).images[0]
              refined_images.append(refined)

          # Rest of the function remains the same
          if not refined_images:
              raise ValueError("No images were generated")

          images = [img.convert('RGB') if isinstance(img, Image.Image) else Image.fromarray(img).convert('RGB')
                  for img in refined_images]

          likelihoods = self.compute_clip_likelihoods(images, prompt_truncated)
          clear_gpu_memory()
          return images, likelihoods

      except Exception as e:
          print(f"Error in generate_images: {str(e)}")
          traceback.print_exc()
          return [], np.array([])

    def compute_clip_likelihoods(self, images, prompt):
        """Compute CLIP likelihoods with proper token handling"""
        try:
            # Ensure prompt is truncated
            prompt_truncated = create_optimized_prompt_for_sd(prompt)

            inputs = self.clip_processor(
                text=[prompt_truncated] * len(images),
                images=images,
                return_tensors="pt",
                padding=True,
                truncation=True,  # Enable truncation
                max_length=77     # Explicit max length
            ).to(self.device)

            with torch.no_grad():
                outputs = self.clip_model(**inputs)
                image_embeds = F.normalize(outputs.image_embeds, p=2, dim=1)
                text_embeds = F.normalize(outputs.text_embeds, p=2, dim=1)
                cosine_similarity = F.cosine_similarity(image_embeds, text_embeds, dim=1)
                likelihoods = (cosine_similarity + 1) / 2
            return likelihoods.cpu().numpy()

        except Exception as e:
            print(f"Error in compute_clip_likelihoods: {str(e)}")
            traceback.print_exc()
            return np.array([0.5] * len(images))  # Return neutral scores on error

    def compute_mean_and_variance(self, images):
        """Original mean and variance computation"""
        if isinstance(images[0], Image.Image):
            images = [np.array(img) for img in images]
        images_array = np.array(images) / 255.0
        mean_image = np.mean(images_array, axis=0)
        variance_image = np.var(images_array, axis=0)
        return mean_image, variance_image

# ImageToVideoPipeline class exactly
class ImageToVideoPipeline:
    def __init__(self, device="cuda"):
        self.device = device
        setup_ffmpeg()

        # Load original transition models
        print("Loading Stable Diffusion XL Img2Img Pipeline for transitions...")
        self.pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16
        ).to(self.device)

        # Enable optimizations
        try:
            self.pipe.enable_attention_slicing(slice_size="auto")
            self.pipe.enable_vae_slicing()
            self.pipe.enable_xformers_memory_efficient_attention()
        except Exception as e:
            print(f"Warning: Could not enable some optimizations: {e}")

    @staticmethod
    def preprocess_image(image):
        """Preprocess image for transition generation"""
        if isinstance(image, str):
            image = Image.open(image)
        elif not isinstance(image, Image.Image):
            image = Image.fromarray(image)

        image = image.convert("RGB")
        image = image.resize((1024, 1024), Image.Resampling.LANCZOS)
        return image

    def generate_transition_frames(self, image_a, image_b, prompt_a, prompt_b, num_frames=15):
        """Generate smooth transition frames between two images using original technology"""
        print(f"Generating {num_frames} transition frames...")

        image_a = self.preprocess_image(image_a)
        image_b = self.preprocess_image(image_b)

        # Truncate prompts more aggressively for transitions
        prompt_a = create_optimized_prompt_for_sd(prompt_a or "a scene", max_tokens=30)
        prompt_b = create_optimized_prompt_for_sd(prompt_b or "another scene", max_tokens=30)

        transition_frames = []

        with torch.no_grad():
            # Enhance source and target images
            print("Enhancing source image...")
            image_a_enhanced = self.pipe(
                image=image_a,
                prompt=prompt_a,
                strength=0.2,
                guidance_scale=7.5,
                num_inference_steps=30,
            ).images[0]

            print("Enhancing target image...")
            image_b_enhanced = self.pipe(
                image=image_b,
                prompt=prompt_b,
                strength=0.2,
                guidance_scale=7.5,
                num_inference_steps=30,
            ).images[0]

            # Generate transition frames
            for i in range(num_frames):
                t = i / (num_frames - 1) if num_frames > 1 else 0

                # Create simple blended prompt (much shorter)
                if i < num_frames / 2:
                    current_prompt = f"{prompt_a}, transition"
                else:
                    current_prompt = f"{prompt_b}, motion"

                current_prompt = create_optimized_prompt_for_sd(current_prompt, max_tokens=25)

                print(f"Generating transition frame {i+1}/{num_frames} (t={t:.2f})...")

                # Create base image for transition
                if i < num_frames / 2:
                    base_image = image_a_enhanced
                    strength = 0.2 + 0.3 * (t * 2)
                else:
                    blend_factor = (t - 0.5) * 2
                    base_image = Image.blend(image_a_enhanced, image_b_enhanced, blend_factor)
                    strength = 0.5 - 0.2 * (1 - blend_factor)

                result = self.pipe(
                    image=base_image,
                    prompt=current_prompt,
                    strength=float(strength),
                    guidance_scale=7.5,
                    num_inference_steps=25,
                ).images[0]

                transition_frames.append(result)

        return transition_frames

    def create_smooth_video_from_images(self, images, scene_texts, prompts, output_path,
                                      fps=8, frames_per_scene=20, transition_frames=15):
        """Create smooth video with transitions using original technology"""
        temp_dir = tempfile.mkdtemp()

        try:
            all_frames = []
            frame_count = 0

            for i, (image, text, prompt) in enumerate(zip(images, scene_texts, prompts)):
                print(f"Processing scene {i+1}/{len(images)}: {text[:50]}...")

                # Hold on the main image for several frames
                main_image = self.preprocess_image(image)

                # Add main scene frames
                for frame_idx in range(frames_per_scene):
                    frame_with_text = main_image.copy()

                    # Add simple subtitle (English only)
                    if text:
                        self.add_simple_subtitle(frame_with_text, text)

                    frame_path = os.path.join(temp_dir, f"frame_{frame_count:06d}.png")
                    frame_with_text.save(frame_path)
                    all_frames.append(frame_path)
                    frame_count += 1

                # Generate transition to next scene (except for last scene)
                if i < len(images) - 1:
                    print(f"Generating transition from scene {i+1} to {i+2}...")

                    transition_frames_list = self.generate_transition_frames(
                        image, images[i+1],
                        prompts[i], prompts[i+1],
                        num_frames=transition_frames
                    )

                    # Add transition frames
                    for t_idx, t_frame in enumerate(transition_frames_list):
                        # Add subtitle to transition frame
                        if i < len(scene_texts) - 1:
                            transition_text = f"{scene_texts[i]} → {scene_texts[i+1]}"
                            self.add_simple_subtitle(t_frame, transition_text)

                        frame_path = os.path.join(temp_dir, f"frame_{frame_count:06d}.png")
                        t_frame.save(frame_path)
                        all_frames.append(frame_path)
                        frame_count += 1

            # Create video using imageio (more reliable than ffmpeg)
            print(f"Creating video with {len(all_frames)} total frames...")

            with imageio.get_writer(output_path, fps=fps, quality=8) as writer:
                for frame_file in tqdm(all_frames, desc="Writing video"):
                    frame = imageio.imread(frame_file)
                    writer.append_data(frame)

            print(f"Video saved to: {output_path}")

        finally:
            pass

    def add_simple_subtitle(self, img_pil, text):
        """Add simple English subtitle to image (no Chinese fonts)"""
        draw = ImageDraw.Draw(img_pil)
        font_size = 36

        # Use default font (works for English)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", font_size)
        except:
            font = ImageFont.load_default()

        # Wrap text
        text_wrapped = self.wrap_text(text, 50)
        lines = text_wrapped.split('\n')
        line_height = 40
        total_height = len(lines) * line_height

        y_start = img_pil.height - total_height - 50

        for line_idx, line in enumerate(lines):
            bbox = draw.textbbox((0, 0), line, font=font)
            text_width = bbox[2] - bbox[0]

            x = (img_pil.width - text_width) // 2
            y = y_start + line_idx * line_height

            # Draw outline
            outline_range = 2
            for dx in range(-outline_range, outline_range + 1):
                for dy in range(-outline_range, outline_range + 1):
                    if dx != 0 or dy != 0:
                        draw.text((x + dx, y + dy), line, font=font, fill=(0, 0, 0))

            # Draw main text
            draw.text((x, y), line, font=font, fill=(255, 255, 255))

    def wrap_text(self, text, max_width):
        """Simple text wrapping"""
        words = text.split()
        lines = []
        current_line = []
        current_length = 0

        for word in words:
            if current_length + len(word) + 1 <= max_width:
                current_line.append(word)
                current_length += len(word) + 1
            else:
                if current_line:
                    lines.append(' '.join(current_line))
                current_line = [word]
                current_length = len(word)

        if current_line:
            lines.append(' '.join(current_line))

        return '\n'.join(lines)

    # Legacy methods for backward compatibility
    def create_video_from_images(self, images, scene_texts, output_path, fps=1, duration_per_scene=4):
        """Legacy method - creates simple video without transitions"""
        temp_dir = tempfile.mkdtemp()

        try:
            frame_files = []
            frame_count = 0

            for i, (image, text) in enumerate(zip(images, scene_texts)):
                # Convert PIL to numpy if needed
                if hasattr(image, 'convert'):
                    image_array = np.array(image.convert('RGB'))
                else:
                    image_array = image

                # Create frames for this scene
                frames_per_scene = int(fps * duration_per_scene)

                for frame_idx in range(frames_per_scene):
                    # Create image with text overlay
                    img_pil = Image.fromarray(image_array)

                    # Add subtitle
                    if text:
                        self.add_simple_subtitle(img_pil, text)

                    # Save frame
                    frame_path = os.path.join(temp_dir, f"frame_{frame_count:06d}.png")
                    img_pil.save(frame_path)
                    frame_files.append(frame_path)
                    frame_count += 1

            # Create video using imageio
            print(f"Creating video with {len(frame_files)} frames...")

            with imageio.get_writer(output_path, fps=fps, quality=8) as writer:
                for frame_file in tqdm(frame_files, desc="Writing video"):
                    frame = imageio.imread(frame_file)
                    writer.append_data(frame)

            print(f"Video saved to: {output_path}")

        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)


# =============================================================================
# NEW: SERVER-READY VIDEO GENERATION CLASS
# =============================================================================

class ServerTextToVideo:
    """Server version of TextToVideoExperiment"""

    def __init__(self):
        print("🚀 Initializing Text-to-Video Server...")
        self.text_analyzer = EnhancedTextAnalyzer()
        self.image_generator = None  # Initialize this when needed (saves memory)
        self.video_pipeline = None
        print("✅ Server initialized!")

    def initialize_ai_models(self):
        """Initialize heavy AI models only when needed"""
        if self.image_generator is None:
            print("🔄 Loading AI models (this takes a few minutes)...")
            self.image_generator = BayesianStableDiffusion()
            self.video_pipeline = ImageToVideoPipeline()
            print("✅ AI models loaded!")

    def generate_video_for_server(self, text, max_scenes=3, use_transitions=True,
                                character_seed=42, task_id=None, progress_callback=None):
        """Server-optimized video generation with progress updates"""

        try:
            def update_progress(progress, message):
                if progress_callback:
                    progress_callback(task_id, progress, message)
                print(f"[Task {task_id}] {progress}% - {message}")

            update_progress(5, "Starting video generation...")

            # Initialize models if needed
            if self.image_generator is None:
                update_progress(10, "Loading AI models...")
                self.initialize_ai_models()

            update_progress(20, "Analyzing text and creating scenes...")

            # Text analysis
            if use_transitions:
                scene_data = self.text_analyzer.analyze_text_to_consistent_prompts(text, max_scenes)
            else:
                scene_data = self.text_analyzer.analyze_text_to_prompts(text, max_scenes)

            update_progress(40, f"Generated {len(scene_data)} scenes. Creating images...")

            # Generate images
            images = []
            scene_texts = []
            prompts = []


            optimal_scale = 9.0  # Default value

            for i, scene_info in enumerate(scene_data):
                scene_progress = 40 + (i / len(scene_data)) * 40  # 40-80%
                update_progress(int(scene_progress), f"Generating image {i+1}/{len(scene_data)}: {scene_info['scene_text'][:50]}...")

                # Use Bayesian optimization for first scene only (to save time)
                if i == 0:
                    update_progress(int(scene_progress), "Optimizing generation parameters...")
                    try:
                        optimal_scale = optimize_guidance_scale(
                            self.image_generator,
                            scene_info['image_prompt'],
                            "low quality, blurry, distorted, deformed",
                            num_samples=2  # Reduced for server speed
                        )
                    except Exception as e:
                        print(f"Optimization failed, using default: {e}")
                        optimal_scale = 9.0

                # Use consistent seed
                scene_seed = character_seed + (i * 5)

                generated_images, likelihoods = self.image_generator.generate_images(
                    scene_info['image_prompt'],
                    negative_prompt="low quality, blurry, distorted, deformed, ugly, bad anatomy",
                    num_samples=3,  # Reduced for server performance
                    guidance_scale=optimal_scale,  # Use optimized value
                    seed=scene_seed
                )



                if generated_images and len(likelihoods) > 0:
                    best_idx = np.argmax(likelihoods)
                    best_image = generated_images[best_idx]

                    images.append(best_image)
                    scene_texts.append(scene_info['scene_text'])
                    prompts.append(scene_info['image_prompt'])

                    # Save individual scene image
                    scene_image_path = f"scene_{task_id}_{i+1}.png"
                    best_image.save(scene_image_path)
                else:
                    update_progress(0, f"Failed to generate image for scene {i+1}")
                    return None

            update_progress(85, "Creating final video...")

            # Create video
            output_filename = f"video_{task_id}.mp4"

            if use_transitions and len(images) > 1:
                self.video_pipeline.create_smooth_video_from_images(
                    images=images,
                    scene_texts=scene_texts,
                    prompts=prompts,
                    output_path=output_filename,
                    fps=8,
                    frames_per_scene=16,  # Reduced for faster processing
                    transition_frames=8
                )
            else:
                self.video_pipeline.create_video_from_images(
                    images=images,
                    scene_texts=scene_texts,
                    output_path=output_filename,
                    fps=1,
                    duration_per_scene=3
                )

            update_progress(100, "Video generation completed!")

            # Return result info
            return {
                'success': True,
                'video_file': output_filename,
                'scenes_generated': len(images),
                'character_seed': character_seed,
                'total_time': time.time()
            }

        except Exception as e:
            error_msg = f"Error generating video: {str(e)}"
            update_progress(0, error_msg)
            traceback.print_exc()
            return {'success': False, 'error': error_msg}



def optimize_guidance_scale(model, prompt, negative_prompt="", num_samples=3):
    """Bayesian optimization with proper error handling"""

    if BAYESIAN_AVAILABLE:
        def objective(guidance_scale):
            try:
                images, likelihoods = model.generate_images(
                    prompt,
                    negative_prompt=negative_prompt,
                    num_samples=num_samples,
                    guidance_scale=guidance_scale
                )
                return np.mean(likelihoods) if len(likelihoods) > 0 else 0.0
            except Exception as e:
                print(f"Error in objective function: {str(e)}")
                return 0.0

        try:
            optimizer = BayesianOptimization(
                f=objective,
                pbounds={"guidance_scale": (7.0, 12.0)},
                random_state=42,
                verbose=0
            )

            optimizer.maximize(
                init_points=2,
                n_iter=3
            )
            print(f"🔍 Bayesian optimization found optimal guidance scale: {optimizer.max['params']['guidance_scale']:.2f}")
            return optimizer.max['params']['guidance_scale']
        except Exception as e:
            print(f"Error in Bayesian optimization: {str(e)}")
            return 9.0
    else:
        # Fallback: simple grid search
        print("🔍 Using fallback optimization method...")
        scales_to_test = [7.5, 9.0, 10.5]
        best_scale = 9.0
        best_score = 0.0

        for scale in scales_to_test:
            try:
                images, likelihoods = model.generate_images(
                    prompt,
                    negative_prompt=negative_prompt,
                    num_samples=2,
                    guidance_scale=scale
                )
                score = np.mean(likelihoods) if len(likelihoods) > 0 else 0.0
                print(f"Scale {scale}: Score {score:.3f}")

                if score > best_score:
                    best_score = score
                    best_scale = scale
            except Exception as e:
                print(f"Error testing scale {scale}: {e}")
                continue

        print(f"🔍 Fallback optimization found best guidance scale: {best_scale}")
        return best_scale

# =============================================================================
# FLASK SERVER SETUP
# =============================================================================

# Create Flask app
app = Flask(__name__)
CORS(app)  # Allow requests from Android app

# Global variables
video_generator = ServerTextToVideo()
generation_tasks = {}

def update_task_progress(task_id, progress, message):
    """Update progress for a specific task"""
    if task_id in generation_tasks:
        generation_tasks[task_id]['progress'] = progress
        generation_tasks[task_id]['message'] = message
        generation_tasks[task_id]['last_updated'] = datetime.now().isoformat()

def background_video_generation(task_id, text, max_scenes, use_transitions, character_seed):
    """Run video generation in background thread"""
    try:
        generation_tasks[task_id]['status'] = 'processing'

        result = video_generator.generate_video_for_server(
            text=text,
            max_scenes=max_scenes,
            use_transitions=use_transitions,
            character_seed=character_seed,
            task_id=task_id,
            progress_callback=update_task_progress
        )

        if result and result.get('success'):
            generation_tasks[task_id]['status'] = 'completed'
            generation_tasks[task_id]['video_file'] = result['video_file']
            generation_tasks[task_id]['scenes_generated'] = result['scenes_generated']
        else:
            generation_tasks[task_id]['status'] = 'failed'
            generation_tasks[task_id]['error'] = result.get('error', 'Unknown error')

    except Exception as e:
        generation_tasks[task_id]['status'] = 'failed'
        generation_tasks[task_id]['error'] = str(e)
        traceback.print_exc()

# =============================================================================
# API ENDPOINTS (WHAT ANDROID APP WILL CALL)
# =============================================================================

@app.route('/')
def home():
    """Test page to see if server is working"""
    return f'''
    <h1>🎬 Text-to-Video Server is Running!</h1>
    <p><strong>Server Status:</strong> ✅ Online</p>
    <p><strong>Time:</strong> {datetime.now()}</p>
    <p><strong>Active Tasks:</strong> {len(generation_tasks)}</p>
    <p><strong>GPU Available:</strong> {torch.cuda.is_available()}</p>
    <p><strong>AI Models:</strong> {"✅ Loaded" if video_generator.image_generator else "⏳ Not loaded yet"}</p>

    <h2>Test the API:</h2>
    <form action="/api/generate" method="get">
        <input type="text" name="text" placeholder="Enter text to generate video" style="width: 300px;">
        <button type="submit">Test Generate</button>
    </form>
    '''

@app.route('/api/test')
def test_api():
    """Simple test endpoint"""
    return jsonify({
        'status': 'success',
        'message': 'API is working!',
        'server_time': datetime.now().isoformat(),
        'gpu_available': torch.cuda.is_available(),
        'active_tasks': len(generation_tasks)
    })

@app.route('/api/generate', methods=['POST', 'GET'])
def generate_video_api():
    """Main endpoint to generate videos"""
    try:
        # Handle both POST (from Android) and GET (for testing)
        if request.method == 'POST':
            data = request.get_json()
            text = data.get('text', '')
            max_scenes = data.get('scenes', 3)
            use_transitions = data.get('use_transitions', True)
            character_seed = data.get('character_seed', 42)
        else:  # GET for testing
            text = request.args.get('text', 'A cat playing in a garden')
            max_scenes = int(request.args.get('scenes', 3))
            use_transitions = request.args.get('transitions', 'true').lower() == 'true'
            character_seed = int(request.args.get('seed', 42))

        if not text.strip():
            return jsonify({'success': False, 'error': 'No text provided'})

        # Create unique task ID
        task_id = f"{int(time.time() * 1000)}_{hash(text) % 10000}"

        # Initialize task
        generation_tasks[task_id] = {
            'task_id': task_id,
            'status': 'queued',
            'progress': 0,
            'message': 'Task queued for processing...',
            'text': text,
            'max_scenes': max_scenes,
            'use_transitions': use_transitions,
            'character_seed': character_seed,
            'created_at': datetime.now().isoformat(),
            'last_updated': datetime.now().isoformat()
        }

        # Start generation in background
        thread = threading.Thread(
            target=background_video_generation,
            args=(task_id, text, max_scenes, use_transitions, character_seed)
        )
        thread.daemon = True
        thread.start()

        return jsonify({
            'success': True,
            'task_id': task_id,
            'message': 'Video generation started!',
            'estimated_time': '5-10 minutes'
        })

    except Exception as e:
        print(f"Error in generate_video_api: {e}")
        traceback.print_exc()
        return jsonify({'success': False, 'error': str(e)})

@app.route('/api/status/<task_id>')
def get_task_status(task_id):
    """Get the status of a video generation task"""
    task = generation_tasks.get(task_id)

    if not task:
        return jsonify({'error': 'Task not found'}), 404

    return jsonify(task)

@app.route('/api/download/<task_id>')
def download_video(task_id):
    """Download the generated video"""
    task = generation_tasks.get(task_id)

    if not task:
        return jsonify({'error': 'Task not found'}), 404

    if task['status'] != 'completed':
        return jsonify({'error': 'Video not ready yet'}), 400

    video_file = task.get('video_file')
    if video_file and os.path.exists(video_file):
        return send_file(video_file, as_attachment=True, download_name=f"generated_video_{task_id}.mp4")
    else:
        return jsonify({'error': 'Video file not found'}), 404

@app.route('/api/scene/<task_id>/<int:scene_number>')
def download_scene_image(task_id, scene_number):
    """Download individual scene image"""
    scene_file = f"scene_{task_id}_{scene_number}.png"
    if os.path.exists(scene_file):
        return send_file(scene_file, as_attachment=True)
    else:
        return jsonify({'error': 'Scene image not found'}), 404

@app.route('/api/tasks')
def list_tasks():
    """List all tasks (for debugging)"""
    return jsonify({
        'total_tasks': len(generation_tasks),
        'tasks': list(generation_tasks.values())
    })

# =============================================================================
# START THE SERVER
# =============================================================================

def start_server():
    """Start the Flask server with ngrok tunnel"""
    print("🚀 Starting Text-to-Video Server...")

    # Start Flask in background
    def run_flask():
        app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

    server_thread = threading.Thread(target=run_flask)
    server_thread.daemon = True
    server_thread.start()

    # Wait a moment for Flask to start
    time.sleep(3)

    # Create public tunnel with ngrok
    try:
        from pyngrok import conf
        conf.get_default().auth_token = CONFIG["NGROK_AUTH_TOKEN"]
        public_url = ngrok.connect(5000)

        # Extract clean string URL from ngrok response
        if hasattr(public_url, 'public_url'):
            clean_url = public_url.public_url
        else:
            clean_url = str(public_url)
            if 'NgrokTunnel:' in clean_url:
                import re as _re
                match = _re.search(r'"(https://[^"]+)"', clean_url)
                if match:
                    clean_url = match.group(1)

        # Save URL to file for reference
        with open('/content/server_url.txt', 'w') as f:
            f.write(clean_url)

        print(f"\n{'='*60}")
        print(f"  ✅ SERVER IS ONLINE")
        print(f"{'='*60}")
        print(f"")
        print(f"  📱 COPY THIS URL INTO YOUR ANDROID APP:")
        print(f"")
        print(f"  ┌─────────────────────────────────────────────────────┐")
        print(f"  │  {clean_url:<51}  │")
        print(f"  └─────────────────────────────────────────────────────┘")
        print(f"")
        print(f"{'='*60}")
        print(f"  📋 Other endpoints (for reference only):")
        print(f"     • Browser test:  {clean_url}/")
        print(f"     • API test:      {clean_url}/api/test")
        print(f"     • Generate:      {clean_url}/api/generate")
        print(f"     • Job status:    {clean_url}/api/status/<task_id>")
        print(f"     • Download:      {clean_url}/api/download/<task_id>")
        print(f"{'='*60}")
        print(f"  ⏳ Keep this cell running to keep the server online.")
        print(f"  🛑 Interrupt the cell to stop the server.")
        print(f"{'='*60}\n")

        return clean_url

    except Exception as e:
        print(f"❌ Error creating public URL: {e}")
        print("🏠 Server running locally only: http://localhost:5000")
        return "http://localhost:5000"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Bayesian optimization available
🚀 Initializing Text-to-Video Server...
✅ Server initialized!


In [ ]:
# =============================================================================
# PATCH CELL — run after Cell 2, before start_server()
# =============================================================================

import os, re, traceback as _traceback
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("✅ PYTORCH_ALLOC_CONF=expandable_segments:True set")


# ── Patch 1: BayesianStableDiffusion.__init__ (CPU offload) ──────────────
def _init_bayesian_sd_fixed(self,
                             model_id="stabilityai/stable-diffusion-xl-base-1.0",
                             num_inference_steps=50,
                             clip_model_name="openai/clip-vit-base-patch32"):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    self.model_id = model_id
    self.refiner_id = "stabilityai/stable-diffusion-xl-refiner-1.0"

    print(f"Initializing models on device: {self.device}")
    if torch.cuda.is_available():
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        free  = total - torch.cuda.memory_allocated() / 1e9
        print(f"Total CUDA memory: {total:.2f} GB  |  Free: {free:.2f} GB")
        clear_gpu_memory()

    try:
        print(f"Loading base model {model_id}...")
        self.base = DiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True
        )
        self.base.scheduler = EulerDiscreteScheduler.from_config(
            self.base.scheduler.config, use_karras_sigmas=True
        )
        self.base.enable_model_cpu_offload()
        try:
            self.base.enable_attention_slicing(slice_size="auto")
            self.base.vae.enable_slicing()
        except Exception as e:
            print(f"Warning: minor base optimization skipped: {e}")

        print("Loading refiner model...")
        # Do NOT share vae/text_encoder_2 with cpu_offload — causes device conflicts
        self.refiner = DiffusionPipeline.from_pretrained(
            self.refiner_id,
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True
        )
        self.refiner.scheduler = EulerDiscreteScheduler.from_config(
            self.refiner.scheduler.config, use_karras_sigmas=True
        )
        self.refiner.enable_model_cpu_offload()
        try:
            self.refiner.enable_attention_slicing(slice_size="auto")
            self.refiner.vae.enable_slicing()
        except Exception as e:
            print(f"Warning: minor refiner optimization skipped: {e}")

        print("Loading CLIP model...")
        self.num_inference_steps = num_inference_steps
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(self.device)
        self.clip_model.eval()

        print("✅ Model initialization completed (CPU offload mode)")

    except Exception as e:
        print(f"❌ Error initializing model: {str(e)}")
        traceback.print_exc()
        raise


# ── Patch 2: BayesianStableDiffusion.generate_images (CPU generators) ────
def _generate_images_fixed(self, prompt, negative_prompt="", num_samples=5,
                            guidance_scale=9, temperature=1.0, seed=None):
    try:
        clear_gpu_memory()

        # CPU generators required when enable_model_cpu_offload() is active
        if seed is not None:
            torch.manual_seed(seed)
            np.random.seed(seed)
            generator = torch.Generator("cpu").manual_seed(seed)
        else:
            generator = None

        prompt_truncated          = create_optimized_prompt_for_sd(prompt)
        negative_prompt_truncated = create_optimized_prompt_for_sd(negative_prompt)

        print(f"Generating {num_samples} images with seed: {seed}")
        print(f"Prompt: {prompt_truncated[:80]}...")

        base_images = self.base(
            prompt=[prompt_truncated] * num_samples,
            negative_prompt=[negative_prompt_truncated] * num_samples,
            num_inference_steps=30,
            denoising_end=0.8,
            guidance_scale=guidance_scale,
            width=1024,
            height=1024,
            generator=generator,
        ).images

        refined_images = []
        for i, base_image in enumerate(base_images):
            refiner_generator = (
                torch.Generator("cpu").manual_seed(seed + i)
                if seed is not None else None
            )
            refined = self.refiner(
                prompt=prompt_truncated,
                negative_prompt=negative_prompt_truncated,
                image=base_image,
                num_inference_steps=20,
                denoising_start=0.8,
                guidance_scale=guidance_scale,
                generator=refiner_generator,
            ).images[0]
            refined_images.append(refined)

        if not refined_images:
            raise ValueError("No images were generated")

        images = [
            img.convert('RGB') if isinstance(img, Image.Image)
            else Image.fromarray(img).convert('RGB')
            for img in refined_images
        ]

        likelihoods = self.compute_clip_likelihoods(images, prompt_truncated)
        clear_gpu_memory()
        return images, likelihoods

    except Exception as e:
        print(f"❌ Error in generate_images: {str(e)}")
        traceback.print_exc()
        return [], np.array([])


# ── Patch 3: ImageToVideoPipeline.__init__ (CPU offload) ─────────────────
def _init_video_pipeline_fixed(self, device="cuda"):
    self.device = device
    setup_ffmpeg()

    print("Loading Stable Diffusion XL Img2Img Pipeline for transitions...")
    self.pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16
    )
    self.pipe.enable_model_cpu_offload()
    try:
        self.pipe.enable_attention_slicing(slice_size="auto")
        self.pipe.vae.enable_slicing()
    except Exception as e:
        print(f"Warning: minor transition optimization skipped: {e}")

    print("✅ Transition pipeline loaded (CPU offload mode)")


# ── Patch 4: segment_text_intelligently (robust parsing + fallbacks) ──────
def _segment_text_intelligently_fixed(self, text, max_scenes=6):
    if not text or not text.strip():
        print("⚠️ Empty text — generating placeholder scenes")
        return [
            f"A cinematic scene depicting an interesting moment, scene {i+1}"
            for i in range(max_scenes)
        ]

    prompt = f"""Please analyze this text and convert it into {max_scenes} distinct visual scenes for video generation.

Input text: "{text}"

Requirements:
1. Create exactly {max_scenes} scenes that tell a coherent story
2. Each scene should feature THE SAME MAIN CHARACTER/SUBJECT throughout
3. Maintain visual consistency - same person, same general setting/world
4. Create a logical progression that flows smoothly between scenes
5. Each scene should be 1-2 sentences describing a specific visual moment
6. Focus on concrete, filmable actions and settings

IMPORTANT - use EXACTLY this output format, no markdown, no bullet points:
Scene 1: [visual description]
Scene 2: [visual description]
...up to Scene {max_scenes}

Example:
Scene 1: A young athletic man in blue athletic wear stands ready on a vast green grassland
Scene 2: The same man launches powerfully into the air, arms spread wide
Scene 3: The man lands gracefully back on the green grass, slightly crouched"""

    try:
        print(f"📡 Calling DeepSeek API (max_scenes={max_scenes}, text_len={len(text)})...")
        response = self.client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": prompt}]
        )

        raw = response.choices[0].message.content.strip()
        print(f"📝 Raw DeepSeek response:\n{'-'*40}\n{raw}\n{'-'*40}")

        scenes = []

        for line in raw.split('\n'):
            line = line.strip()
            if not line:
                continue
            # Format 1: "Scene 1: description"
            m = re.match(r'^Scene\s+\d+\s*:\s*(.+)', line, re.IGNORECASE)
            if m:
                scenes.append(m.group(1).strip())
                continue
            # Format 2: "**Scene 1:** description"
            m = re.match(r'^\*\*Scene\s+\d+\s*:\*\*\s*(.+)', line, re.IGNORECASE)
            if m:
                scenes.append(m.group(1).strip())
                continue
            # Format 3: "**Scene 1: description**"
            m = re.match(r'^\*\*Scene\s+\d+\s*:\s*(.+?)\*\*\s*$', line, re.IGNORECASE)
            if m:
                scenes.append(m.group(1).strip())
                continue
            # Format 4: "1. description" or "1) description"
            m = re.match(r'^\d+[\.\)]\s+(.+)', line)
            if m:
                candidate = m.group(1).strip()
                if len(candidate) > 20 and not candidate.lower().startswith('scene'):
                    scenes.append(candidate)

        print(f"✅ Parsed {len(scenes)} scenes with primary regex")

        # Fallback 1: any sufficiently long line
        if not scenes:
            print("⚠️ Primary parsing found 0 scenes — trying line-length fallback...")
            scenes = [
                l.strip() for l in raw.split('\n')
                if l.strip() and len(l.strip()) > 30
            ][:max_scenes]
            print(f"   Fallback 1 extracted {len(scenes)} lines")

        # Fallback 2: chunk the original text
        if not scenes:
            print("⚠️ All parsing failed — chunking original text as scenes")
            words = text.split()
            chunk_size = max(len(words) // max_scenes, 5)
            for i in range(max_scenes):
                part = ' '.join(words[i * chunk_size:(i + 1) * chunk_size])
                scenes.append(
                    f"A cinematic scene depicting: {part}"
                    if part
                    else f"A cinematic continuation of the story, scene {i + 1}"
                )

        return scenes[:max_scenes]

    except Exception as e:
        print(f"❌ DeepSeek API FAILED: {type(e).__name__}: {e}")
        _traceback.print_exc()
        return [
            f"A cinematic scene depicting: {text[:80].strip()}, scene {i+1} of {max_scenes}"
            for i in range(max_scenes)
        ]


# ── Apply all patches ──────────────────────────────────────────────────────
BayesianStableDiffusion.__init__              = _init_bayesian_sd_fixed
BayesianStableDiffusion.generate_images      = _generate_images_fixed
ImageToVideoPipeline.__init__                = _init_video_pipeline_fixed
EnhancedTextAnalyzer.segment_text_intelligently = _segment_text_intelligently_fixed

print("✅ All patches applied — peak VRAM is now ~4-6 GB instead of ~22+ GB")
print("   Next: run start_server()")

✅ PYTORCH_ALLOC_CONF=expandable_segments:True set
✅ All patches applied — peak VRAM is now ~4-6 GB instead of ~22+ GB
   Next: run start_server()


In [5]:
# =============================================================================
# CELL 4 — START THE SERVER (run this last)
# =============================================================================

setup_ffmpeg()
public_url = start_server()

print("✅ Server is running!")
print("💡 Keep this cell running to keep your server online")
print("🔄 Your Android app can now connect to generate videos!")

try:
    while True:
        time.sleep(60)
        print(f"⏰ Server running... Active tasks: {len(generation_tasks)}")
except KeyboardInterrupt:
    print("🛑 Server stopped")
    ngrok.kill()

FFmpeg is already installed
🚀 Starting Text-to-Video Server...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



  ✅ SERVER IS ONLINE

  📱 COPY THIS URL INTO YOUR ANDROID APP:

  ┌─────────────────────────────────────────────────────┐
  │  https://tuberculous-lura-relativistic.ngrok-free.dev  │
  └─────────────────────────────────────────────────────┘

  📋 Other endpoints (for reference only):
     • Browser test:  https://tuberculous-lura-relativistic.ngrok-free.dev/
     • API test:      https://tuberculous-lura-relativistic.ngrok-free.dev/api/test
     • Generate:      https://tuberculous-lura-relativistic.ngrok-free.dev/api/generate
     • Job status:    https://tuberculous-lura-relativistic.ngrok-free.dev/api/status/<task_id>
     • Download:      https://tuberculous-lura-relativistic.ngrok-free.dev/api/download/<task_id>
  ⏳ Keep this cell running to keep the server online.
  🛑 Interrupt the cell to stop the server.

✅ Server is running!
💡 Keep this cell running to keep your server online
🔄 Your Android app can now connect to generate videos!
⏰ Server running... Active tasks: 0


INFO:werkzeug:127.0.0.1 - - [21/May/2026 14:07:23] "GET /api/test HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/May/2026 14:07:25] "POST /api/generate HTTP/1.1" 200 -


[Task 1779372445518_8] 5% - Starting video generation...
[Task 1779372445518_8] 10% - Loading AI models...
🔄 Loading AI models (this takes a few minutes)...
Initializing models on device: cuda
Total CUDA memory: 42.41 GB  |  Free: 42.41 GB
Loading base model stabilityai/stable-diffusion-xl-base-1.0...


...

Video saved to: video_1779372445518_8.mp4
[Task 1779372445518_8] 100% - Video generation completed!


INFO:werkzeug:127.0.0.1 - - [21/May/2026 14:13:57] "GET /api/status/1779372445518_8 HTTP/1.1" 200 -


⏰ Server running... Active tasks: 1


INFO:werkzeug:127.0.0.1 - - [21/May/2026 14:14:14] "GET /api/download/1779372445518_8 HTTP/1.1" 200 -


🛑 Server stopped
